In [1]:
# --- New architecture: put the standalone src/ modules on the import path ----
# The aligner code now lives in src/ and uses flat imports (e.g.
# `from servodriver import Servoset`), the same as clip_scan.py /
# calibrate_jacobian.py -- not the old `expctl.servers.servoaligner.*` package
# paths. Add src/ to sys.path so this notebook (under example/notebooks/) can
# import those modules directly, without the full expctl install.
import os, sys

_src = None
for _base in [os.getcwd()] + [os.path.abspath(p) for p in ("..", "../..", "../../..")]:
    if os.path.isfile(os.path.join(_base, "src", "servodriver.py")):
        _src = os.path.join(_base, "src")
        break
    if os.path.isfile(os.path.join(_base, "servodriver.py")):
        _src = _base
        break
if _src is None:
    raise RuntimeError(
        "Could not locate the servo_aligner src/ directory -- set `_src` manually."
    )
if _src not in sys.path:
    sys.path.insert(0, _src)
print("servo_aligner src on path:", _src)

servo_aligner src on path: /home/mmwave-piservo-1/servo_aligner/src


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time, pickle
import logging

logging.basicConfig(
    level=logging.INFO,  # Set the logging level to DEBUG
    # level= logging.DEBUG,|
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# New architecture: flat imports from src/ (see the path-bootstrap cell above),
# matching clip_scan.py / calibrate_jacobian.py.
from scservo_sdk import *                    # FEETECH SCServo SDK library
from servodriver import Servoset
from servo_util import (
    create_zigzag_X, format_para, a2p, r2nd, r2nr,
    ndmodr, nrselr, nrmodr, nraddr, compose_para,
)
from fit_gaussian import (
    gaussian_2d, gaussian_2d_smooth_heaviside, fit_and_plot,
    fit_gaussian_2d, fit_gaussian_2d_smooth_heaviside,
    fit_and_plot_smooth_heaviside, popt_get_mu_cov, statistics_skewness,
)
from motor_scan import motor_1d_scan, motor_2d_scan
from step_optimize import pts_iterator, step_optimize
from config import (
    MASKS, knob_mask, posmask2str,
    SERVER, SERVO_CHANNEL_LIST, ACCEPT_FUNCTIONS,
)
# The photodiode ADC and the optimizer-callback factory now live in
# callback_functions.py (the old pd.py was merged in). intensity_adc is the
# default objective; MCP3424_fiber is the raw ADC handle the logging cells use.
from callback_functions import make_callback_func, intensity_adc, MCP3424_fiber

# local aliases for the per-path knob masks, so the cells below stay terse
A_X_XDOT_MASK, A_Y_YDOT_MASK, A_X_Y_MASK, A_XDOT_YDOT_MASK, A_POS_ALL_MASK = (knob_mask('A', g) for g in ['X_XDOT','Y_YDOT','X_Y','XDOT_YDOT','POS_ALL'])
B_X_XDOT_MASK, B_Y_YDOT_MASK, B_X_Y_MASK, B_XDOT_YDOT_MASK, B_POS_ALL_MASK = (knob_mask('B', g) for g in ['X_XDOT','Y_YDOT','X_Y','XDOT_YDOT','POS_ALL'])
POS_ALL_MASK = MASKS['POS_ALL']

# Build the Servoset from machine.yaml (board id + channel list). NB: the new
# Servoset does NOT energise the motors on construction -- enable holding torque
# explicitly, the way clip_scan.py / calibrate_jacobian.py do.
servos = Servoset(board_id=SERVER["board_id"], servo_channel_list=SERVO_CHANNEL_LIST)
servos.torques_enable()

# The optimizer callback (compose para -> move servos -> read objective) is now
# built by make_callback_func, replacing the hand-written callback_func this
# notebook used to define. Same signature: callback_func(para, pos_mask,
# zero=None, jac=None, jac_master_mask=None, debug=False, **kwargs).
callback_func = make_callback_func(servos, intensity_adc)


# Beam-clip accept functions now live in calibration.yaml (ACCEPT_FUNCTIONS),
# keyed by mask name -- the same source clip_scan.py reads.
def accept_func_linearxy(xy, slope, b, tol):
    x, y = xy[0], xy[1]
    return np.abs(y - x * slope - b) < tol

def posmask2acceptfunc(posmask):
    name = posmask2str(posmask)
    params = ACCEPT_FUNCTIONS.get(name)
    if params is None:
        raise ValueError("No accept function configured for posmask {}".format(name))
    return lambda xy: accept_func_linearxy(xy, params["slope"], params["b"], params["tol"])

2026-06-29 18:19:31,073 - INFO - Succeeded to open the port
2026-06-29 18:19:31,090 - INFO - Succeeded to change the baudrate
2026-06-29 18:19:31,093 - INFO - Servo 1x: SCServo acc set!
2026-06-29 18:19:31,097 - INFO - Servo 1x: SCServo speed set!
2026-06-29 18:19:31,102 - INFO - Servo 2x: SCServo acc set!
2026-06-29 18:19:31,106 - INFO - Servo 2x: SCServo speed set!
2026-06-29 18:19:31,108 - INFO - loading position from disk
2026-06-29 18:19:31,109 - INFO - Loaded, the positions are: 
Servo 1x: -0.087890625 deg	Servo 2x: -106.875 deg	


## 2D Scan

In [ ]:
# accept_func_linearxy / posmask2acceptfunc are defined in the setup cell and
# read their coefficients from calibration.yaml (ACCEPT_FUNCTIONS), so there is
# no need to redeclare them here. To override the config values ad-hoc for a
# scan, redefine posmask2acceptfunc, e.g.:
#
# _overrides = {
#     'A_X_XDOT': dict(slope=1.35,  b=0, tol=80),
#     'A_Y_YDOT': dict(slope=-1.36, b=5, tol=80),
#     'B_X_XDOT': dict(slope=-0.58, b=0, tol=60),
#     'B_Y_YDOT': dict(slope=-0.67, b=5, tol=80),
# }
# def posmask2acceptfunc(posmask):
#     p = _overrides[posmask2str(posmask)]
#     return lambda xy: accept_func_linearxy(xy, p['slope'], p['b'], p['tol'])

In [ ]:
def slope2perpvec(slope):
    return np.array([slope,-1])

In [3]:
N_pts = 25
POS_MASK = B_XDOT_YDOT_MASK
SCAN_RANGE = 50
center_zero = [0,0,0,0,0,0,0,0]
cf = lambda para: callback_func(para,pos_mask=POS_MASK, zero=center_zero)
# cf = lambda para: callback_func(para,pos_mask=POS_MASK)
X1,Y1,Z1 = motor_2d_scan(N_pts,SCAN_RANGE,servos,cf)
# X1,Y1,Z1 = motor_2d_scan(N_pts,SCAN_RANGE,servos,cf, accept_func = posmask2acceptfunc(POS_MASK))
# np.savez("/home/rydpiservo/servodata/servosetup1/scan_AXXDOT_pm10+rand10+lin70.npz",X1=X1,Y1=Y1,Z1=Z1)

2026-06-29 18:20:02,803 - ERROR - Goal position list length 8 does not match the number of motors 2


OSError: [Errno 121] Remote I/O error

In [ ]:
popt = fit_and_plot_smooth_heaviside(X1,Y1,Z1)
mu,cov = popt_get_mu_cov(popt)
print(mu, cov)

In [ ]:
popt = fit_and_plot_smooth_heaviside(X1,Y1,Z1)
mu,cov = popt_get_mu_cov(popt)
print(mu)

In [ ]:
# jac_x0_pm11_rand11 = np.load("/home/rydpiservo/servodata/servosetup5/jac_pm11+rand11.npz",allow_pickle=True)
# jac_pm11_rand11 = jac_x0_pm11_rand11['jac']
# print(jac_pm11_rand11)
jac_x0_rand11 = np.load("/home/rydpiservo/servodata/servosetup5/jac_rand_11.npz",allow_pickle=True)
jac_rand11 = jac_x0_rand11['jac']
print(jac_rand11)
# jac_x0_pm11_rand11_lin20 = np.load("/home/rydpiservo/servodata/servosetup5/jac_pm11+rand11+lin20.npz",allow_pickle=True)
# jac_pm11_rand11_lin20 = jac_x0_pm11_rand11_lin20['jac']

In [ ]:
N_pts = 15
POS_MASK = B_XDOT_YDOT_MASK #3x,4x
SCAN_RANGE = 35
cf = lambda para: callback_func(para,pos_mask=POS_MASK,jac=jac_rand11,jac_master_mask=B_POS_ALL_MASK)
# cf = lambda para: callback_func(para,pos_mask=POS_MASK)
X1,Y1,Z1 = motor_2d_scan(N_pts,SCAN_RANGE,servos,cf)